# Esercizio 1 — Object Detection YOLO (demo)

Dimostrazione del detector **YOLO** su un subset di **Object365** (8 classi: Person, Chair, Table, Cabinet, Car, Lamp, Picture, Monitor).

Tutta la logica di detection è **implementata a mano**: griglia di celle, anchor box, costruzione dei target, **loss YOLO**, **IoU** e **non-max suppression**. Il solo componente importato è il backbone pre-addestrato su ImageNet (**ResNet34**, *transfer learning*), che viene fine-tunato. Con `PRETRAINED_BACKBONE = False` in `config.py` si usa invece un backbone convoluzionale scritto interamente da zero.

Il notebook mostra le predizioni con bounding box, la valutazione (**precision, recall, F1 e mAP@0.5**) e la detection su una foto a scelta.

Risultato del modello finale: **mAP@0.5 = 0,444 sul test set** (0,465 sul validation). Richiede un modello addestrato in `outputs/best.pt`, prodotto con `python src/train.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import torch
from config import (DEVICE, CLASS_NAMES, OUTPUT_DIR, PRETRAINED_BACKBONE, BACKBONE,
                    GRID, IMG_SIZE, STRIDE, EVAL_MIN_OBJ_SIZE, load_anchors)
from model import YOLO
from detect import detect, draw

print('Device:', DEVICE)
print('Classi:', CLASS_NAMES)
print('Risoluzione / stride:', f'{IMG_SIZE}px / {STRIDE}  ->  griglia {GRID}x{GRID}')
print('Backbone:', f'{BACKBONE} pre-addestrata (transfer learning)' if PRETRAINED_BACKBONE
      else 'convoluzionale scritto da zero')
print('Compito: oggetti con lato minore >=', round(EVAL_MIN_OBJ_SIZE, 3), 'del lato immagine')

## 1. Carico il modello addestrato

In [ ]:
from config import check_checkpoint_config

anchors = load_anchors()
model = YOLO().to(DEVICE)
ckpt = torch.load(Path(OUTPUT_DIR) / 'best.pt', map_location=DEVICE)
check_checkpoint_config(ckpt)
model.load_state_dict(ckpt['model'])
model.eval()
CONF = ckpt.get('best_conf', 0.25)  # soglia che massimizza l'F1 sul validation set
print('Modello caricato (epoch', ckpt['epoch'] + 1, ', mAP@0.5 validazione',
      round(ckpt.get('best_map50', float('nan')), 3), ', soglia', CONF, ')')

## 2. Detection su alcune immagini di validazione

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from dataset import Object365Detection

val = Object365Detection('val')
sample_ids = val.ids[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, name in zip(axes.ravel(), sample_ids):
    img = Image.open(val.img_dir / f'{name}.jpg').convert('RGB')
    results = detect(model, img, anchors, DEVICE, conf_thresh=CONF, nms_thresh=0.45)
    ax.imshow(draw(img, results))
    ax.set_title(f'{len(results)} oggetti')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Valutazione quantitativa

Precision, recall e F1 alla soglia salvata nel checkpoint, più **AP per classe e mAP@0.5** (indipendenti dalla soglia). Il validation set è quello usato per scegliere il checkpoint; il test set, valutato una sola volta, dà il risultato finale: `python src/evaluate.py --split test`.

In [ ]:
from evaluate import evaluate

precision, recall = evaluate(str(Path(OUTPUT_DIR) / 'best.pt'), conf_thresh=CONF)

## 4. Detection su una tua foto

Imposta `MY_PHOTO` sul percorso di una foto (es. scattata con la fotocamera) contenente uno degli oggetti tra: Person, Chair, Table, Cabinet, Car, Lamp, Picture, Monitor. Il detector è pensato per oggetti medio-grandi (lato minore almeno 1/20 dell'immagine).

In [ ]:
from PIL import ImageOps

# metti qui il percorso di una tua foto; le immagini in data/images/camera/ non sono
# mai state viste dal modello
MY_PHOTO = next((ROOT / 'data' / 'images' / 'camera').glob('*'), None)

if MY_PHOTO:
    # exif_transpose applica la rotazione salvata nei metadati delle foto del telefono
    img = ImageOps.exif_transpose(Image.open(MY_PHOTO)).convert('RGB')
    results = detect(model, img, anchors, DEVICE, conf_thresh=CONF, nms_thresh=0.45)
    print(MY_PHOTO.name, '->', len(results), 'oggetti')
    for cls, score, box in results:
        print(f'  {CLASS_NAMES[cls]:8s} {score:.2f}')
    plt.figure(figsize=(9, 9)); plt.imshow(draw(img, results)); plt.axis('off'); plt.show()
else:
    print('Nessuna foto in data/images/camera: imposta MY_PHOTO con il percorso di una foto.')